# Reusable GPU runtime — URL and private token
Select a GPU runtime and run the cell. It starts a token-protected localhost execution service behind a temporary HTTPS tunnel. The token grants runtime code, shell, terminal and file access. Google Drive is not mounted.
Node, Chrome and FFmpeg are cached once per runtime for the permanent illustrated WebGL cartoon pipeline. No scene, character asset or video is installed or rendered during setup. NVIDIA rendering is enforced when capture starts.
A live dashboard refreshes every 3 seconds with completed frames, worker activity, system RAM, stage, measured throughput and ETA. WebGL supports 1–8 browser workers; native H.264 capture uses one persistent browser by default because it was fastest on the measured two-core T4. More workers do not guarantee more speed.
Portrait and landscape full-HD project formats are supported. The runtime includes frame-preserving audio/video finalization; completed saved segments can be finalized without rendering them again through the local engine recovery command. Frame counts, stream hashes and black-frame checks remain enforced.
Rerunning in the same live VM reuses the authenticated server/jobs and refreshes its tunnel. A new/deleted runtime has a new token and loses ephemeral files. Colab limits and external-control restrictions still apply; uptime and free GPU assignment are not guaranteed.
The character style remains the existing illustrated round-headed family; richer details do not replace it.

In [ ]:
# Reusable runtime bridge: run once, then send jobs through the URL/token.
import base64,hashlib,io,json,os,pathlib,platform,re,secrets,socket,subprocess,sys,threading,time,urllib.request,zipfile
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi>=0.110,<1','uvicorn>=0.24,<1','psutil>=5.9,<8','requests>=2.31,<3'],check=True)
import requests,psutil
PAYLOAD = 'UEsDBBQAAAAIAK4gal2WLWeuiwEAAHkCAAAbAAAAdmlkZW9fZW5naW5lL2ZyYW1lX2NhY2hlLnB5ZVFNj5swEL3zK0ZcbKuUAoqqLCt6qLqtctluVamXsqIODBsLYqOxabKq+t9rDOqHevDYM37z5uPFcfxpVu0APckzwnc5qk46ZTRI3UE7otTzBITWkWodduAMGI1wMTQgMQvmon2UpH7CNI7jSJ0nQw48fm5d1JM5wyTdaVRH2L4evBtFHfZrtWbST3yBiDICR8/ewkW5U8CtH6mZUHNGRyZA2oUb5XnBwQllh1StkdSbjhc7cbv5FnHgL/MiKcQt6u4fWF4IT0DoZtIbzddy/1hVR1Zf9zcP9x9qqnV9zWWtWVjGOlI660m2A2dvDgeWbIn567LYPYqq4nm2z5L8pshEyFmqBsYs+/sc7u7f1VeJb7/5WgWLAK8tTg74x893RIaSL3KccX1uZXFxRLk1/F6OFtclBo2aIEMTRLS8U4StM/Sc9IqsS0Zp3bLd3tCms9KrZPwP4EW+QIJYVVj9bxbxqmchrfkRrjLbdT9TL5tvHFQfUlJlm16NyNe5tXH/qRtgsx6VHriIfgFQSwMEFAAAAAgA8R1vXYbJLJx5AwAAggYAAB0AAAB2aWRlb19lbmdpbmUvcnVudGltZV9zZXR1cC5weY1UUW/bNhB+N+D/QPiFUiEr6IC9KNCDtxpBsLYL3HTZUAwELZ1sJhKpkpQTd9h/30dRTlxkA/YgiDzyjnf3ffctFoufZbUn5vG5oe+N9VSzO9pevWfemLbaS6WZ0RVdMm2Yq0gTM5bBbmXlyTKlnZdtK70yOl8sFvOZ6kIYdu+Mztyw7a2pyLn5rLGmY730+1Zt2XTpBtvpxA7aq47EI213rXDkh/50K26kY2dnk1e09NJ6FTI4eXxY/S7u1j9dvRd3v25+WW8+zWfzWU1NDJVYY3xazGcsLMqQRDTllpxpD5Skl2Gfdw+1sgmik/auvLUDZfSknBfmYdylCNFbpX3CN58/3l5/WItPt6urNYwEL6V3Uzmhl45nTTu4/bPnaCzPaopJ4GTXD+VL63K0JvnC9UHVSi5dp3jGl8uvA9njMtzUsqOstupAVuBz6EPWUWfsMfcG4IzXG2M76cvKHTJt9iRrsvzPrJK9HywJM/h+8LFCT0+nFfDASfn2x5CUakJeaBE8dGVqKqxUjtgmAre21tiErzT7+Nv1u+sVu7r5fAKVKccsfR2UBbuQCTYaCaA/OQ+hgylUATaxhKPse6o8+sWBx9BilRZjby7CpXSC5RUSKEZuVQsikCv/evNm7C9iwJEXztvY3uwl/rPx4sWGc1TJi1Cq8zXKx8+qPkkzPMF4zFxsZfWAFS/4iB9SfR4J4fyxJZz0hJZrUIepFsB7K8NwVeCqAVMb2an2yGPUyI/iC++Pfm80wrk9tSFso1oScNWuAWIZ3w6qrcU4iHwsZUuisWhLbNeYHIhCBuiOoTv5JCb7o7EP4AcvXo1HxjEcEq1+dfVtFisU/y/O+GR0sFSZrgsetdDQhwN9FxWp7wCvE1h4U5l2tD6LUCxCRNrG1sAOxnnU2UpdO8BNpyKdPLy4KC1b9W1UJF6MROY7K/u9qlyYD9WoajrkE1VjXUHNHCONNysA9bgnzSSbRoTdmy2D1Fkk8zfenHgzEXx5zr08SB9P80erPLDDOCXBktdD17vk/Gam8K725Q/pvyjJZr169wfP/sM1PReTyziU301AUDyMrBBhZoQoSy6AodJC8KB8k1BKu4NQOQrPl6dNvrK7Acj5m7CzUMM+l3Ut5GROoCfjVGUTa0p+URntcXQRMSC9UyBoeinLPh9jBmeHSFHoZD5J3T9QSwMEFAAAAAgAeYVqXZB/eYBmBgAARRAAACAAAAB2aWRlb19lbmdpbmUvcnVudGltZV9wcm9ncmVzcy5weY0X226bSPTdXzHKy0BKiNNd7YNdKnVVt6rUTSMnu/tgWYjAYFNjYGeGJFWUf99zGTCOnTYvMJw59zsnJyc3a62S7MwkuRK5TrZK2NompRFJlYmtSkyrVSbsWtftat20VuS1Fo3SpjBWVVZoVWVKi/tabwAYnpycjIptU2sr0rosVWqLujLBd1NXQW0CS9KKahXYYqtGua63oknsuixuhSO7gk++IHXiNEnXqru8S8oii5tqxRi6rZBPvFXbWv/okNa1sQ40GqVlYoyYk5ZXul5pZcxkJDKVizguqsLGsWdUmQe6rm1AIk2QF9rYAChtkJZ1uolQSritq9rWVZH6wEAgUYhEEWrs4cmfEpCZMJjP3QWyjejJAJQQ4YM/WRY9OwHGJtqqjIGe48Mqda4M518Hdy4Q0eMTf68hTuCIaBCNMFP/taojWDVtdFlXqpOY1tumVCjzMadg56KohE6qlfJ2fnlz4Ysi38XDG1h+nksO3WM+Gf+ePYVwL/2nXgAGNItKVXn7An3ASOvKqgcb9d49l42LWohJJAHH6h8TR8nIeAFOSTLjOVCIvonx5PnIVj2kqrHC+3Y907rWwT9J2So6+vusHp84N7RaYYJrzo0yuVVlcFu3VWYo+PeFXYs+FpOh5xeEvIweZZHJCVNKcpycMIfFeBlIdGIPuABAg+gYiEBC0G1r5ERCmFqVSbhcJ0YB4N+ksBByqFFBmQFX5GqmdLqrOyjMoeIEOKY3QFzlRkdMwNvCsVtIesllFJF6IBGvBZ0jh8IXyymzCdsmg28PDIvoJlwp65GdfsAmMrwn9IkliBygQ3pKFyPMVEeA0OURbMiVFFLFVT8QsioL6QDLjsEzxH1evUKY+Z7sEhSdnRQlBGTHlwOzjOQn6CUG8loecgGX9SyEKo0S8hPzQakIYF+WRdW7Es87rfCLW4HBCHry0/zDX7P4+ubD/Gb2UUifGVBl7DufsiMqIB+YR1MWUBKQcH5AqkfySqsm0ZhU566Z45mzigPSV8+uaoIvgPjgCqiBDkuIqjym659/f/n6EXX9PINen25gnCRVsU2wFx3z5Nea2powqQIf/IQxO+Fq/u3zfHZ9ve+FF+12R3kufajD1xmolW11xaigSKcxF/XyXUTC3kUdnEp7SQOUJyqMDcyk/W5H97/qn/ga9FBn4DNOYZJlHGh/r+eHSdNAQD1vN1w8PzjSd7FFYuMwVdKYde16B7h6BfnR54R8qYNU9X00FDGlDSLqJ9zZbvi9uZim0PLsC91f3K+hLkR/6Qzx378lbw1hEEqI3zuQffbHeLJndlMDxxzCjRzzxtB4myqbdHOOSupARvR24pINvAADak/aGYk724ONETTNVAmcD5AvDpEv+np2It5HF2MyjHigAqgtfZw7FFLbI4eekef8c8BxNrArI7qdIOJ419BNtCBxOMEZ0qdgt6vdYbYbz2VVYdXWpTAEpfkBcUTQou9dGXbPdusdpv9B6vdrw5EQC+ZKSgPHfcqzZ8whXzptXTIjNfHhohSPp6ePm70ZvtiwAhvu3tDpv8PiA81b49oDq2GBE9UmZrODqSxmOXFZbAtETjJ47+AGQA7jts1g1sSwYeA+QmvQZmAr6fAUkKmSakhO6BXsPBlzocsJRTBgZ/RA+oKZr3SKI3eicUnwLsbjU0I/5/sL38ngfaqnHuxYYAVsfHeqt2FC8Vt0CwZOJlihKyxuThS0wyH7h84BBlg2PQLL7+/cIZAud2OjwB2Z6SzAWh0utWgCdzoTg7EOvcOGNA9+I+/CiccmbUcsFLL9OXsAETq8RWGo6SLBgFQO/gzkZPABbZFWin7PCCTPT9CGVn98eL7br5r2toRZ/7MuCftqW7otdtdVERW6Y6Ihf17ecKeQ4fAfk8DSzqgh9tzYtHlePHgytNuGBnOPFt5rqAredmkVztptYzzWIShAMegSb31/xxjW46ZMUuWxAOTWaByWcj67/DibD4bqm0OOBheGBLqaiTwoiwD2Ut8P8rI16+hGt8of9dXJFKMRraVQUHFnKv0vsfOC01PuROi55z9U+Ay3m6zQHggFQwyJCNQD9NW43rDAzqfHfxhwH0hbjdTDXwXn3P0/hZ/9KHQ88B/Bnbs1i7OAzXGZEw0SZ2fiSLwmutNfxdaJf2VwXWxnl5+/XM5eiG3H8RXBdbF1FKP/AVBLAwQUAAAACAB5hWpdlxJ+UWUFAAAACwAAIQAAAHZpZGVvX2VuZ2luZS9ydW50aW1lX2Rhc2hib2FyZC5weW1WXY7bNhB+9ylYFQEtrCPbSVos5B8gRZKiRVIEmwJ9WCwMWhxZykqiSlL2LgwDPUTv0vcepSfpDEnJ3iQv3h1yOD/fzHyjKIo+1aKqWKMsbJW6Z1KYYquElgn7VW1RtIKVhoHJRAtywTINEhpbisowoYE1sAfNyiarOgkyiaJoVNat0pYVtq4mtqxhNJKQM9lpYUvVjPei6iBOR6zMmfuf7P+mGkg12E43jIPBZ6jc7PiIGchUI82qFg/j2aRsbDAQL+qy6SyYSa8hy32t5DiIkx9n8aJQnTaToNgr9O9QYcSCz5wfne6pYMdwn85eyFPNjsGeEw2nqJ0mg8oAvQvqX6v6vDU0EvRmwHX8WW0Npe8hXVWi3krhkUgJs8RfjI3VfaojluFLs7q9G7FcaYYmEHI0jdgb8CZvn1+nd2SXtSuUkx3YMW+12mkwhscMnx1PC2OF7cxZwct8wrvmvlGHhqMvShH7gbVMNJJ5je9WXHdNQyVJMUesTQeoiZc7WLWDrR2gKfyrqXgUJjpGa8Hr2YYHzx+jmYPS95hJbyiIfHJ7Fy8y1TW2v8lU3VZgQW5yLWpAFSyyVVZUvYYTLm4JD9AZtmzoICzXeD6bTfJKCTsOr4IKPYgJbkZpeX3q4IR+xvHz9pZ3LY4ESH7nEgvP+8PY50XNjDa0OpgV58lnhS750uo1v+rFHGXC9rGCVdQKKRGV9FX7wOYv2odofQw9sI9Py6mVa+7Kvseio13GxodbXmIMk8OTKsa93BbCwFl0aPgW4P/99Tef5NHx8AWglPpp2h87FN1RNNgs5YWJOL7iGJkOkR2oHUPZCD4JVpTVCqfDlQ8NO4sn5gvD/v2HHQPmaTLPT8/8ifMUicyWe9gEc5EPLNxp+LNDesAGuLxm/kkfgRtS59DXg3/U0ApNLWmVQuKaMoOuYeHDCSNK14iba33XwkhnyD8Ma49XmMvbSrQ4bew4UFkICvzFJox/FGNEF/n4nDeYb9CIHI7RmeaiHpipg+YGalHSnITIKAT4hlsrLl1yP7luCqkcYw5NpqixNnX7CueyrzWNaJdlgESOJU1DfvydL02Onk2BuTN+NeZvgwlHBVkB2b1DsQD24eOrhA8OcbafeAvA0wrpDS6Y6mzbWaJ7scf2ENsK0JRW3a5wJre6lDtIHAehVRyNnBMe74WxPRR+1NiRFgH6jU9obKdcIC5vXz+aQBcynq3nP1wQ2VckxAm3GmqlH3sOKZSxG3/UE+egs7HwYFd59OkRu7BmN68/pLgx3FXgAuqEXYnMd8I+e3LluSnc/Vz+xEh38VRnQOZSbzhMIkrU64fwz4EwCxXUYPGqRbAwRde/oLXSfWpO6EneCef6o49OQ4p1p9XjLuOh/69WHC8uMPAmnjD2Ztth/exGgzCqGVrLP0Xu/KYWrjS/3RLRUthEjbiov+TGOZLjYqs09kA6R540qiol+/76+jqcPtdClp1Jr1GvFnpXNul8hoqzaL3cDoQaRgf34mfIbDTBFXUb4c+mlNHdbTp/cUfDiwzHjsMadm22nG7Xy3aw4wmOjls8DlvWL/FV1FPbKWK4Q1YRrpuoz+dQSltgZLNnGNe0f3lpOoDmBiA4sFT7NW2QpS3WfzgYkX0LJ35yfT2IH4n7B8kN9SC9wbE4K/7yxv9PNH6kZUW7xnuaYgXW/Px55EqyXhYv1++JZ3GCHCXRd8dyiqdLQ1+R6xvIMZsCWYS+TB7Zy/7rLWFvf39Nw1lj0bHJcDBzi9OcX3LOgpWWwYP7knRnxN+BVKYSP05wY8tkOfW+iJ3CMnXt43cT4ngDQj66Eh5E6b5DaEMJCjYhNLlbXS7B0f9QSwMEFAAAAAgAUIVqXQ2uDSyuAwAAQAgAAB4AAAB2aWRlb19lbmdpbmUvcnVudGltZV9tZW1vcnkucHmNVdtu4zYQfddXDNwHSYEsX7ZFATt+2ADt7qLrbhEX6UNgBLQ0itlIpEpSSQzD/97hxbKzKZLqiRwezuUMz2gwGHyW2sD1xyWwsuFacymAiRIM1tigUbs53NhTroGBxpYpZhA2XXmPJh8MBhFvWqkMSB1VSjbQMrOt+QaC+Q/aRtGnL1eLyXj648XFhygqsYItBb1rsJFql6SzCB5Z3aFe7A8RUEgyQCUV1FwgcOGcJPGoVbIY0SUuKhmnuUJW3hl8Nkma67bmxsK1cwcgWIOZQm0W1urPk3gWZ5N07oPdWsh6wYVJLC5A0tvxOr2wuZIXIw2rFwEeL7H50xri9Zw9Ml6zTY3hMCcyEgv4eDyIs+9OflVIxnGaRoDPBbYGkm+rX5SSKruxSL/8DXdukc4Umk4J+F0KjOAHeFE8FExAibpQfINgtuj4hKctUu/AIlFr2DLXsobVNSoo7pXsWqK04SbvWXZbx0HgWO/0qNIjjx75DuUNe37JNxUBRacUivcvB9xrB7yC8aVL4NIRPfN0O8sZw1Rx0u8ySiUZZw4zDJ7fpjSdtUzrCAKf+9hFudvsDOrYh8zi3v/RfgoYdxrLo9lHd5eGPSTN7IPzX/B+zzfxjEgQZeIMIxJANk3PA51BemMPczHPEMn3IQPy4NUUOn4UVMtLK4LQ4jNpvRZVFVS1t1209w4jbZjp9Hv6ouZ5YRmmjH7i5CuJb5rr1Yo0RovPfy1nceqx8IC7d7RIiLelGNrn4W/3+9jpQM6TVA+o7vzEIv//kAODZebJWliB2TTD9sVgslX6tR1/FglYawy2yB4LacL2GPcUIe5HayG7unTYjb3NdKewnJ+gIUmLE8bWyoioMo+t8pcBTs3r1LCQ7Q5WQj5BRW+B7GDnNUym+c/wiV/BzTKHr8geEaZubzv+bZVZRxQeN1I+ZFAq/kjR3JhXTGhOKgIaE7Jghua/tplpVOTkQ/6Tc9MSXBdIBNgEaH6wijyXTqC2bV4YnojbV3JaD6cX9GDT0Sghh26ZOvZOXi4nM8U4cXtNDPDG9zKpBl+E7qqKFy7FbeBzBvvXoaxc1geXbW+cg+3ZVY2i7AsQiCUNRgHEPG+YZb+vsu7syPTMBQpyWBnZgqQpq0Knh1sieAd/y42mflIFaFumdlzc5wMqTNOvsyC/bnSdPTjiaJKdSrYckMa0FIvqP/7C/7vI7GXCWV/OqUDbviML/qnNYd9ndjh7s7A/Zn9wqRha5YN+fB4PM5949C9QSwMEFAAAAAgA8b5uXUi5r4tRCwAAhBsAAB0AAAB2aWRlb19lbmdpbmUvcnVudGltZV93ZWJnbC5wea1ZbW/bOBL+7l9B9IBSamnlZbuLPRv6kG3dbrBp4nXcdu8MQ6Al2lYjSzpRspPt5b/fDIeU7cTJdoEr0ESihpz3Z2aYFy9efBh+YuNlpVTwVbNYlnVTKVYvq6JZLOG3YlquFJtVabJQgm3SesnWqkrnqUpY0dRlU7OVzNO50rUOXrx40UlXZVHVTFaLUlZaibjIMhXXaZFreM7jpqpUXgfzBjlpMY/zOhNLqZdZOhNfdZGLQotKCd3MyqqIldaiTldKgExKJmm+6MyrYsVKWeMOZtkN4ZU+VE2O9BFsXgAD7SjUKq3bxX3SjZotskiruikdtXnZp1qpVVHdOYJloWu7RGR0COhcp6iso2sXxNDyFh/P/oi+DH75cBF9uRr9NhhddzqJmrMkXYARPVTM73XI1KgWrQRFqXKPVzPuM6mZrsEaq14FYlY5s+YL5mmmInsOUQiul/L0x5+4HyzVrf3kE0NQzKuKoha5RPtKfSPSBHyDzq3EpqhuVKXDUxQGyUIjDD75AehRZGsFJ7F0zvIC/J1lXqXArVm2knUMhHxy1v237P553P1n1J2+5mLts3lRsTVLc+YZnlt2vt+rZKoV+yyzRg2qqqg8fp6vZZYmDLz2FULoqFL/aUB+dv4u4MDZLoco0hG3b5of4dEdlhXxTei5rTywbux+LWYBfuPOpBLPqqu7ngnFYI4fPfxBsRlcXL39LRr88d+dt8tfYIu6jVVZs1+QFMLy/MpI3cPXIM4KDdbpk04jYm21Gi9T7WRHzzGZMwkJslYMZUNp0PZWLRAbVNoa6oiv4aWIVL5Ic8X7SBqsbpK0gjjBzNLhuGqUULcpBGhxY96sgh22nwUebhZgOQVbQQVKBC6cYY2PrNGjNAl3ogOjJcQfcDSriyLToUkZio++ytdhoQP4lVZFHsRFeYexgsHikV6LsunazyvEA8x8cEmqIwxiz+/Bx6ApE1krD7+Bz2Sived3I0JEtbqFEPeR3UzGNypPwt0D2ogAYyWqiizNoSPQXlsp3v46uvo4CI2yEx4DRK4Un4rR4PLdYBQBjob8hMPr75/OR4Po8vP5u/Mzs/R+eB3yH465+P06dNwWqva4hdsIDFzdccFfglPCk5dShm9eQtomxSY8PX7zMzeS6LhKyzpsxTdyHEGgNUlkDwpWXzUHUkzuuFgBLifeqkiUeAVorDGPGQZeHG6xNRiaJJhYrXKgBp0AOzzi5ws6YCVLRBRhDpqKeJM4SQT6Gv7DpgQKwt7Z58MBLquq2l2+Hr+7+jQWaGSK1Vkz1+mfKjxBPQ1GZBDbCBO4JaCDjfQYQPgN1gBXNWKkx60HhqOrD6PB9TXjpClja8QSvet92ltmKVifcXHiT06mfh8OW6iQqIOyKD1uVrjf312jMEi4uCxyBSH+OJPMLvHqFe36q8wROyUk3HmmwIPQy7TqlZCXtRFbYCBzLuZZo5cuqxn4OVGhMdNGprVJMjQSLvcOoM+cf1GzDxfsG7r1HjAsBZWo1uAW9g1/3veh/ilT/QEbYoVVF1qAYqMBf1SOJBVTeJ5BKwYWaLI6/OYwmPcMcnDUkveMrnxrDN7bsQanHkLz3mR6TwBhDBTyWZNmQEs6UixzUw7ACYaryrbElMtE/Q8GlXamGPQaID9ICqgq2ExBXCmWySaPlzv66D6WLtB3gUCiWaNxpzLWiOk4YM3K9FZlGJpQ3AI2uIUQitOaXX4eXL5lVjfN5jLNWAVlErjWS8B1DVgGVeOOxfAGgL0I8EhrQjCYfeI9bsXpLk9/etPVxbzeAJoDJhjGETEGsrtm/eb0uORoUImI1ePYwYGFykJT42GcmUtTUOypxBHY96FWM7AAkGPYgut/HY+HUBxkrucgxz064CBGGRLsaLgP9oZCYQykuQlTtpNiO4keL1V8E5GHD0HMQyx7yMrgmY8FCvzJxZ5crgqS/QSXTV3wJ5CpRRrfNxXo+/huaxHpuOstaLZuwVHf5Z9MLWR811Z8y41jtDODG0fu4C4ASN1oV4o2FSQn1SJj36RZldqztCLNMYugPfP7BBIcKs7gKoKAvAI4ZPz1403+Q/TYKW97u0O7YdLqPBVEMDz/Y3ARvb8afTwbb6n2rOBI7VnR58Ho/P354J0phm/PhuNPUB/Ho7PL6+HVaHy4JG7DzbnWyEuxFJLVHHQcmWbyNe+a9iVYlW8AueljQD2RbZAe9kSg/20JPlFPdAhgF3mE2KsPdQcTnjSQ65ByfNqvi1pm0LI1AFPu0Fc/HBsmOALosB0EPEPr+mv0HtWQ0A0IlsBsM58xmaCOfbvvxxJALQvbWSgYrEE9Qn1T9cl8HgbHrZihOLbuA1JWYOHQ2gVBIkKzQUUgSbrfzKZ7az8AySyJbAbZBm67OLHBdnV58S8+RccaHn+7uaA05maNBPTNM0k+OZ4egQ13V07MyuM0byX7v7UhrDX8xBhmaqps6y3sQGpnaOzSc3CHaZ3joEwTOgKLJ+TozggcJFgrvJW8hcIQnv5IdAayCQvm3ERx94FTsmLRDiwbLhyeh7yp592faSYEGtv4PNtCObECWcJxiWkufPsFjiDYodU+vhvI8BzFdzZgrZksupCd/qoTc0xMaX+Sza9n14M9HiVMUX+fRdtdzfmEbG6NPQXsfKbjetRyPRETWGQzBc2K68fC8JgqCbSXBdg+oS8mpbHYwADlag3HTgIIfNf0mn6ORLcbDDUAAjQsYY6IZPWtIbM8fy8EyN94DKcZleOq8bCYc+wBd5s/y5Q90z1ScDJrrx5sRDFs2/hU/AbcJYW5s6B0d8CFFolKBYDoWbxy9ye6qdOs46IazU7C41WJy9CAOm4AZWsjUJUyETIPjAEDNzbtLv7tLGz+FZsc4J+4BEM60GuzmBAPSkho6AL36oFJm0pDi7UTF1ZEQ0NmX4M+KvHcLr/lakQwy0GtqlWaY9y0p7irBSsV+ayUWu8JfWDnc/tsj//wAi4Ym1oyBIge3IJSNbgZ0Cly1z8AUx5VIoMxJdCRFnZ/OMGlAHAV5iHP1h+RutpjLOLe0Cgqb1aqQqHp1GkfQQihDAPaHkr6bN20AUthL28InRET8Khwmw8oZpLTfjdXiFgWfhAUftFmCU49sOv9+eh6HL29+ji8GIwH75xlUQ2YI1AD5NvDmxoae7bGP3C1QnPJztXKq1dbsMhlqZdF3c579j4JsGxgHqFeHEr53VTpP6W3M+ShMdWByxPC0KUVMnZDUTiZt8oaS8zRDpbF1MzseANKLTk1gW36Uo2OErB/rrECtuR0YTqHCM6g6roN7t2Q2WfPCSL2W7NVegvarrmgpkZQ3+TaL/GYt2vtfNGmmq301IFlcjVLJDNzfO+ZAf+Q2YwTyUqTtjWdhhOYimEg5D1sYmjZby9le/ZCtl2f3dUKZmHbpGE58Xz4FWFfIvgNgCkMFNQz3U93Ge6PQ1PXmffd98d9NdJAH/eoH+fTjq2Ojy9kCf1twMFUq5sSz9LMDOsMoi43bY7MmB3V4cVnEDGUCFQCbCdq7zeNgNjEb28FXnPbb/eJ9Lku3pEcHpbo9J1Z6Ykr0LZWC7qJbeHvuL0Hn0zbQWtw+eH8chBBr/PpYrw/aBHDh3OWLXn0sXMw0xFczZXKobhrM9ZQhBhK5sl/JG6bvSZ3sru96+hOB+piZNr+KIIpPopWMs2jCO9NytD9xSY4qxYNptsQ3yrsbwKZJJG0yx7vdtF33FxwpRWUT/LEYzJ3H/TXlOaqSMTLIo2xptjbHwegMDVAkyDxksmtHJSK7mG6afIdHK3JuKjvShWCX1vukCIL5Z08/jvN6xO/leMU/CrDMjAmw4MRjfEvKjIwf1ORgRtPZGBuwGC9vQKDFzf+df4HUEsDBBQAAAAIAPEdb11NEqALIQQAANwIAAAjAAAAdmlkZW9fZW5naW5lL3J1bnRpbWVfd2ViZ2xfc2V0dXAucHmtVV1P5DYUfZ9f4fJih804u6jlYVAe6JbSlVi6gu6qFUKRJ7mTGBw72M4wtOp/77WdGRZaUSr1JfHHvcf34/h4b2/vvag7IGa1krUUipybBojQDXnfWdPjhiaCnEk9bsjpp8/EjtrLHnJyL31nRk/qTuhW6pY0Vq7BOr63tzeT/WAs7vmHARwfvVR5J1yn5DK/cUbnxuWui8tuXA7W1OBc7oVdSQX5aBUacgt3Izg/W2EYZBA+eJMJ+BNOZ7MGVsSBHwdmjfHZYkbCvwybaeUofHl/20jLBmFBe1f+YkfIYSOdr8xtnGVH3hjlymBc0DimaWlyfWo9I3JFtPFEKMVSFvy+k3XHdEZWxhJNpCaMrlb9AC3NcYAZLoFmGQm7k+dXteErqZsK07PCPjxDOTk9Qwj8ZCE/8lgujp1gV1QMft6CR5v53R1+x6ERHuh1jk2tdxH/u5/UzmNYYf4QY56Cx6igVe/SaDfYYC0bsNuZPJwGt8va9L3Rb9Pc9YfPI1mjo7GuVFhSFjtFi9HZwnXYn6JVa90UeGCV7HhDM94qs2R0X69lI8U+DwTCarx5jqBMLdR/x5khhRTUHpryD1pVp2e/Vl9Ozn/4+aI6+/D9xfHFb9X58ccTuqDJj/4Z+z+lsdj6XqErtuq5648fzk6C+yW9LhGC3xipWS8G5rzNJ4wQgnEcEN8azVMD2RY4O2KJl+0wziebHok8hc/vrfRQedh4FlZ4M/aD23nnSCw0Lg/CIRpvtisd3h9oWOJ3KknYmK8PDvj+XIWLPt8cflsspS7CRixRZDx6L8K3jMOr+bvrGQHlIPByKRyUtPN+cIsi+t04bmxbNNilQmFGzscjNgWNdOxd+fSec5yaATQLUG/o5U/Hl58/Xh58d8j9BmkaRAflpjx8m6GHaFjGG6jxHBboDZsh5atFj/GFamAmwN2gpGfpQoWFcKfC2WkjrDjcxeyiNfbDBVlj9LEOHFWJb36PVSDC1h2KXBmLV4Szjv6ehAVvJawhJRKM8skvQAT8LQ6P+VK7pBkRjiAnQPQLUftRqHLSSx4ksWpki+gsWeQUSY6Fwe53sJm2AjbmkZy/KbcFWVghHZAvQo1wYq2xjEaBH8Yl3p8OLMFCzxGM9NL1wtcdp7swJ0FOYW5ziJGa0dawSD+O1bZ4bhC1WJgcnTzYkiKRRUSLrHksGlarN2twI745G0Z3JS7ojnQzfFfC81M+EVjaGtMqmKc9GvX0BYM5StpSAX2U7LQeCNvAcoroH32qerThvahE3yAJ0DrQ9oVm77jfKJ4AOUphEXmEl8BiN4pXHpTj51Wa/f9oPZKKhROfw7ymAfjiYhuVR+mMbVsEsDDKcjrZLNJ/96QsngJOqxnK6iR094DvzDzp04sil47+SuKOsBuj1VNMs78AUEsDBBQAAAAIABKTal1FsVnHIQQAACAKAAAfAAAAdmlkZW9fZW5naW5lL3dlYmdsX3BhcnRpdGlvbi5wea1WwW7jNhC96ysIXyjVtBMvUKBQlgu0QJpD03aRRTcLGIZAS5RNRCJVkl4nDfLvHQ4l2Uls9NKLQQ0f38y8mSE9mUyuH0XpSW1FK4nZa2ndVnVE6IqIzcbKjfCSdNbA0jlSG0s6gCjnpfZkbc3eSUv2xj6AdT6ZTBLVdsZ64rdWikrpDfOqlUltTUvsToePopWtsU+kR26N870p+f3nb8X99S83t8X9n3e/Xd994T8lSSVr0gnrlVdGp9540bDeY5YnRNUEbR8XxIyhEG08UZpYoTcyXbB3xNNFlluhnCRfRbOT19Yam9b0Xq5vbsnOSUeMlkBMnt8dfRmdoEqkMw5C+y57EUsDac5plgww3qq3YV859Y9k8tFbwSv1vTXVO4CHjPnlVUjc8eUqQe2VruTjIa8jFYjUFcdD08A9TRH6EV1kkWUuug5QaYowBsts8APrhFjpd1aj1C5JykZAwT/3lQcPoQxFobTyRZE62dQshox4VjamfOChvPPWgPhGqxLjCsg5Ijn+XqEhwvE3GjAOWUVTmkVj5Bw6aX53i3sD6SDv8pmqiuY13cv1ppk9qxfKaK2s8zQX0wWjkAks14yWpu0aCW5ofskohgP2mWAU3Pudozn9eyd3sM9otxVOguHWoPOx10PNXSm1DJjg9w9olJdYHZYKts5CfaTetdLC7KQoT7aK+gU3MoqH9WHRLwMiHnhQsb3yWzLmHyyv8l3iydVyiHnF4yLgYBaAKj8JD8GuOPz2wHiK81EVeuYcCrHi9NfQ3g5SK00FEgUa2YxEIeuU1kI1KJ/zBrqtotl/kH7pcVGfXVe9EUh52Z5TJZLyEw6u4ufyqOIrHMPBHmu/Yq14TE9gwblPg+sja4HjDXpnWXbwvqRoBvrxGztvNT1BO1uMgY0C3Hz+i8RbpxQdDKDspUDA/6jEuTAPmfaIbLY4Tm8INGDmG+nT3sLoHdwa0obZuCAhizCma1E+0Oy106Kf7dCp4+yHMcaJ0KJzW+Mx03PpjRJyt2vT/StNcfL2ofuOs8+uZAPE4cSRy9nxVROIw4TykeyiPxOmo19+4j/ixI+YT/zDJWzCuxHmFTngXsALmsRQMIC38WAaAdrrPy9N94RXGc4ibBzNM6fwUmoQlqJv2IzCv1YT7pn4zIVAcmToazXl9YSkz68yx/035cjni/rFZZMhtvGFgDWGFqeZC/2UHsV3btLPVAI1ik/L8/tpykdLfyGPG4eHAy5aaeHKhUvcwuNapYeKHUA/LC4v2SJjmAyF/zTwIhd9DEAW++aUxKejZtRKeAtcCHVkaaRO36DGvaBf77zvncLJ0ujKDVH35hBkrIQrIK8eNYBCR7IPGTZF+Os1thqj0osDZTAFUCxDhEWG9KDJbBQquwhsJ2mP/n/R/OgjzV6SfwFQSwMEFAAAAAgAzKpuXQfDC6Q4AQAAJgIAABwAAAB2aWRlb19lbmdpbmUvdmlkZW9fZm9ybWF0LnB5bZBBSwMxEIXv+RXDXpKFdm091cIeBBUvUlHpRaSkyeyakk2WJAsF8b+bdNNtD+YQmJnHmzdfURQbg4DHXiuhAjSD1vPnB2is63gAYU1wXIRUQ29dLKKIGwk6fl7wHqF39oAi+KooCqK6pIKDt4Y0znbQ8/Ct1R7y4DWWhDxt3l7uP96hhh96dqVrYMvFajGD5d3topwBnVacRrEZR1FQ/hJCJDY5406qDo1X1nhmeIf1xbFcE4hPNZAGYGwAZSAvH2fpRa1H2HI94KNz1jG6VRLtmUE3+AB7vJwfUUzRKlqefByGwU3en2nfV46Z+VznzK2czwun+hBhJGqVtlx6xhKoSXdDR02VFLSsHHK5C3gMDI2wUpm2pkNo5qu5Vy0tx0inm+vsXrUYGB0vopHthdF1/P+JluQPUEsDBBQAAAAIABQAb11Nk2OwAwgAAGETAAAeAAAAdmlkZW9fZW5naW5lL3dlYmdsX2ZpbmFsaXplLnB5rVhbb9w2Fn73r2CnD5RijTx2ncAdVwVaNGkCZNPsrcHuZCBwJMrDWCK1JOVLjfz3nnMoaca27AUW+5CYFMnD71y/w5nNZm+UFrX6QzInrmTJPsnNr++ZkxeN1N6xa+W3pvPMSl1Kq/QFE/oW/qlGeGU0q6xopEtns9mBalpjPdsKt63VJvnijE6MS6xMXLdprSmkcweVNQ1rhcctrD/xEaYHB6WsGOzayAiXk6qiScb7AY+XB4DCd1YzFJ3WRpQu2olOi60sLnMA23Y+WvXHEj6/4gmX1hoLf+dua65z560UjRvnlbGgDk5NBf+jeJ7AJkISrxMvb3z2D9vJOO5xiuJS+hxVjYS96MhWALlp5QUixr8E+Fv2FvawwjStBYxg3ytVStMLcAnTxrNSFqaU5ZGVc6lpyFpVgKbSMWOZ30r2l4+nIER7obS06QEjRM/ojgDuq/5ih5PPG9HCymJ5tVyg1sUSNxamvcXZxlU03568Os2b9tQbobW82eAamge1xnH/123FyctX+IHvW2p0Fn5KwZiqjeLUtbXyEc94chyvjtfBms50uvQWTJIXtSkuI2+8qNF8qiIDKae080IXMiwlSvuYTIOzH7LF0grlJPtd1J18jfpG/CfWGqe8upK9ySlSwYidRoGA7j+dsrLkgNSJpq2ly0jci+NXi8WIvuIC3EDr2enZYrFIRCvK7HprapnXUmd3/eGviQAdmwzyJO/3j0s86Fn1mRYN2ZU06iYJTkuCYvKmlYWXZVIq2OEgw1wCPr7A4Mk+GC0fBtl0ngwXZCvMraiNUxBg6iu4GkAWvYsYxD1rmdLj/vV5QJPRsTCOzwFl+ACDaUmjp8a68bxzPhhKiVr6seKEQgKHRjOhZ8hdoMb6IIAFFPt4QVWYViZ7XDjic3J7BpHrI0enHR7F7Ss+VIA1AncrjllX5P62lXydZZyO4v0BQAbxFtG39EJC+OpNHuDyZBHH5wFkKtoWnB/RDI+CZBqDCdAcQcCKX6vSbyFV+ulWqout5+v4m8x3YJFo53kK8b1rA0gNF3PYTfnJ0QLssYH/TrX82thLaQdrYdQrPdgdZUMGU3W5tgYKe6iCS8YPx9IXHOu6JmjlECR69bFLq8kr3VC02N2ekK+9s8/ZEO7sjsR+TRlERqAaxEeRML8QgHZz2woHdASIhfeyaeFUih4SXalMvhcEmFNjDIRVBwVWly6rgDN8tDux4n3lX6942VliNPAEKS02Lrp3ej6AjX9Mj6eC+qOVLRSLkmlhgywGWB4YebwGbgn5leIh7dPmslRodkuJi2kF1UA5n5vLIckARyF81p9Das5dV1XqJuLpYPHU33g+7k2vrfIyxxoccZ5+MUpHs0rVkvHZzs2Q020toLryz5+hkB/x3ZcZnyUz/Mxn8SEM9GwyEeOEmAv6g4x3vpqfIQR0k7HC3t5DjPHblxagBdkccrSAV6JOgWzwnNKwBkkf6CYoQlwtKom0hWMVCDosxmty2VAol8Mg4gOqvBf9LftrJ7THdsfo+pYcs6Mf7hiZHNscij3mwFRAO1iWcOsek6AsYquUYRNhvXSeFUKzUrlC2JL21wI/mlY9IH6yoYDE2sxHYRRscLqqpMU7z5mWV5AHHYSZgrYKvqbs9Ixdvv2DHbHvFqxqHcsYstVAYEckLJ0mtL1uwXZ6ukngc+gAXgT774yMZf9x0xCmx0uBU6hCD9sIAc57gtmH7QL2CVFQ00GT4+9PLuksYiG6pZvMVVWLC+zXDiuwKLQC1gdoY4hBl0Y90B4fTYQDbdkLh72yMUr6/xIIrP6PhR/pFHrt6LFwihT+JKJ4ojb9RvkGESXLUI1AYE3Fqurqev72l776w4379YtuSns9/gsTPSCrVt3kVQOlKHRwmkX8trs6PVlg1MDoCw2fwdp0Dp8dDbLH2XwDKfA2hXsg9v/1z9/h8ASs8U4w0ngFqXQPe21sboW+CODbgk+AeINmoU39dQMeYjNFzGYt8AELmxpYKIUXAyqkj0A3zxDNPqWcvJxA8U6PtxApgr+G48NFz7YmT9P17FMg6kZaQF8b0A38V0IgKiioY7kKspbsbvIKYPKjkbpngGcjQU+Z7T+P+mISqg0QcuWlvbdhNdaZ/WTu95OKQeo3GZ2d8lVoGEPIFlv0R/nkk+scTFl1Dkt8KWt4HthbMiUUbZ09WSO3ICLf4DuIqqQGewnv9ohohx0rKvLWBvjzsoROC5rkMluki+Mlxqff4hi8DU+8MPn+jEqeHt5XGkKvf04VosVnYP+wC03B+MS6X/AmmwLCEChmTrvh1QwP0/2uAPUGHobgslMcvoHHzqqPZBEnYbCJw9tBJBvMbCAmYMtS1HVkg9o5lehltPpcpuvDmIWPEMLjJ7Da7uIYvTzc8WP6HaXsMP9hSJJ5+jLwPGCaakB/3ikLh1WB4R3sDzFAsbP7FQMiGqSslmdr6Dn/NhkQxo1N0I4b+ifRE+aGs6rC/MH8pF8R7pmafrYou6Z1VBTgFVtC75SdYJstMUiB9bM7DqRpC5kPrRVfrqhRm3quJcNm0joP/TVfhr8JDyCHbF2OpWK1l8PrwMa9HFrO+/TsX/bLkH+juMk9ZN8k8F7/+0re6T4Z+ZLiFe8Z6liu5fUIbJHwvtUOfQtf9oM+i3r18CHAlxwfAiDz6xNOqPaqwfNOGGx+zxGtRTPxT69//vV9/ubdh5/ev/v36194MnEO06Fz2we/d5Brhy0HfwJQSwECFAAUAAAACACuIGpdli1nrosBAAB5AgAAGwAAAAAAAAAAAAAAtoEAAAAAdmlkZW9fZW5naW5lL2ZyYW1lX2NhY2hlLnB5UEsBAhQAFAAAAAgA8R1vXYbJLJx5AwAAggYAAB0AAAAAAAAAAAAAALaBxAEAAHZpZGVvX2VuZ2luZS9ydW50aW1lX3NldHVwLnB5UEsBAhQAFAAAAAgAeYVqXZB/eYBmBgAARRAAACAAAAAAAAAAAAAAALaBeAUAAHZpZGVvX2VuZ2luZS9ydW50aW1lX3Byb2dyZXNzLnB5UEsBAhQAFAAAAAgAeYVqXZcSflFlBQAAAAsAACEAAAAAAAAAAAAAALaBHAwAAHZpZGVvX2VuZ2luZS9ydW50aW1lX2Rhc2hib2FyZC5weVBLAQIUABQAAAAIAFCFal0Nrg0srgMAAEAIAAAeAAAAAAAAAAAAAAC2gcARAAB2aWRlb19lbmdpbmUvcnVudGltZV9tZW1vcnkucHlQSwECFAAUAAAACADxvm5dSLmvi1ELAACEGwAAHQAAAAAAAAAAAAAAtoGqFQAAdmlkZW9fZW5naW5lL3J1bnRpbWVfd2ViZ2wucHlQSwECFAAUAAAACADxHW9dTRKgCyEEAADcCAAAIwAAAAAAAAAAAAAAtoE2IQAAdmlkZW9fZW5naW5lL3J1bnRpbWVfd2ViZ2xfc2V0dXAucHlQSwECFAAUAAAACAASk2pdRbFZxyEEAAAgCgAAHwAAAAAAAAAAAAAAtoGYJQAAdmlkZW9fZW5naW5lL3dlYmdsX3BhcnRpdGlvbi5weVBLAQIUABQAAAAIAMyqbl0HwwukOAEAACYCAAAcAAAAAAAAAAAAAAC2gfYpAAB2aWRlb19lbmdpbmUvdmlkZW9fZm9ybWF0LnB5UEsBAhQAFAAAAAgAFABvXU2TY7ADCAAAYRMAAB4AAAAAAAAAAAAAALaBaCsAAHZpZGVvX2VuZ2luZS93ZWJnbF9maW5hbGl6ZS5weVBLBQYAAAAACgAKAPwCAACnMwAAAAA='
EXPECTED_SHA256 = '83c0f035837d929b706e9fce265321c48ed4b187d9cf61e51be9f38ae394d22d'
server_source = '"""Colab Codex Bridge: authenticated localhost execution API.\n\nNever expose\nthis server on 0.0.0.0; tunnel localhost only. The bearer token grants remote\nshell execution with the permissions of the Colab runtime.\n"""\nfrom __future__ import annotations\n\nimport base64\nimport hashlib\nimport collections\nimport json\nimport os\nimport pathlib\nimport secrets\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom datetime import datetime, timezone\nfrom typing import Optional\n\nimport psutil\nfrom fastapi import Depends, FastAPI, Header, HTTPException, Query\nfrom pydantic import BaseModel, Field\n\nBRIDGE_DIR = pathlib.Path(os.environ.get(\'COLAB_BRIDGE_DIR\', \'/content/colab_codex_bridge\')).resolve()\nSTATE_DIR = BRIDGE_DIR / \'state\'\nJOB_DIR = BRIDGE_DIR / \'jobs\'\nPERSIST_DIR = pathlib.Path(os.environ.get(\'COLAB_BRIDGE_PERSIST_DIR\', \'\')).expanduser() if os.environ.get(\'COLAB_BRIDGE_PERSIST_DIR\') else None\nTOKEN = os.environ.get(\'COLAB_BRIDGE_TOKEN\', \'\')\nSTARTED_AT = time.time()\nMAX_FILE = 4 * 1024 * 1024\nMAX_LOG_BYTES = 160000\nLOCK = threading.RLock()\nPROCESSES = {}\nJOBS = {}\nEVENTS = collections.deque(maxlen=100)\nTERMINALS = {}\nENGINE_ROOT = pathlib.Path(os.environ.get(\'VIDEO_ENGINE_ROOT\',\'/content/video_engine\')).resolve()\n\ndef job_environment():\n    env={k:v for k,v in os.environ.items() if k!=\'COLAB_BRIDGE_TOKEN\'}\n    config=ENGINE_ROOT/\'gpu-environment.json\'\n    if config.is_file():\n        values=json.loads(config.read_text())\n        env.update({k:v for k,v in values.items() if k in (\'__EGL_VENDOR_LIBRARY_FILENAMES\',\'__GLX_VENDOR_LIBRARY_NAME\',\'LD_LIBRARY_PATH\')})\n    return env\n\nfor d in (BRIDGE_DIR, STATE_DIR, JOB_DIR):\n    d.mkdir(parents=True, exist_ok=True)\n\nif not TOKEN or len(TOKEN) < 32:\n    raise RuntimeError(\'COLAB_BRIDGE_TOKEN must be set to a long random secret\')\n\napp = FastAPI(title=\'Colab Codex Bridge\', docs_url=None, redoc_url=None, openapi_url=None)\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat(timespec=\'seconds\')\n\n\ndef event(event_type: str, **values):\n    with LOCK:\n        EVENTS.append({\'at\': now(), \'event\': event_type, **values})\n\n\ndef put_json(path: pathlib.Path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.\' + secrets.token_hex(4) + \'.tmp\')\n    with temporary.open(\'w\', encoding=\'utf-8\') as f:\n        json.dump(data, f, ensure_ascii=False, indent=2)\n        f.flush()\n    temporary.replace(path)\n\n\ndef secure(authorization: Optional[str] = Header(default=None)):\n    supplied = (authorization or \'\').removeprefix(\'Bearer \').strip()\n    if not secrets.compare_digest(supplied, TOKEN):\n        raise HTTPException(status_code=401, detail=\'Invalid bearer token\')\n\n\ndef gpu_details():\n    command = [\n        \'nvidia-smi\',\n        \'--query-gpu=index,name,driver_version,utilization.gpu,memory.used,memory.total,temperature.gpu\',\n        \'--format=csv,noheader,nounits\'\n    ]\n    try:\n        result = subprocess.run(command, capture_output=True, text=True, timeout=5)\n        if result.returncode != 0:\n            return {\'available\': False, \'error\': result.stderr.strip()[:500]}\n        rows = []\n        for line in result.stdout.splitlines():\n            cols = [x.strip() for x in line.split(\',\')]\n            if len(cols) >= 7:\n                rows.append(dict(zip([\'index\', \'name\', \'driver\', \'utilization_percent\', \'memory_used_mib\',\n                                      \'memory_total_mib\', \'temperature_c\'], cols[:7])))\n        return {\'available\': bool(rows), \'devices\': rows}\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return {\'available\': False, \'error\': f\'{type(exc).__name__}: {exc}\'}\n\n\ndef job_with_progress(job):\n    result=job.copy()\n    path=JOB_DIR/job[\'job_id\']/\'output.log\'\n    try:\n        with path.open(\'rb\') as stream:\n            stream.seek(0,2);size=stream.tell();stream.seek(max(0,size-262144))\n            lines=stream.read().split(b\'\\n\')[:-1]\n        for line in reversed(lines):\n            if line.startswith((b\'RENDER_PROGRESS \',b\'ENGINE_PROGRESS \')):\n                try:progress=json.loads(line.split(b\' \',1)[1])\n                except (ValueError,UnicodeError):continue\n                if isinstance(progress,dict):result[\'progress\']=progress;break\n    except OSError:pass\n    return result\n\n\ndef status():\n    ram = psutil.virtual_memory()\n    disk = psutil.disk_usage(str(BRIDGE_DIR))\n    with LOCK:\n        running = [j[\'job_id\'] for j in JOBS.values() if j[\'status\'] == \'running\']\n        recent = list(JOBS.values())[-12:]\n        events = list(EVENTS)[-15:]\n    return {\n        \'online\': True,\n        \'timestamp_utc\': now(),\n        \'uptime_seconds\': round(time.time() - STARTED_AT),\n        \'hostname\': os.uname().nodename if hasattr(os, \'uname\') else \'unknown\',\n        \'python\': sys.version.split()[0],\n        \'gpu\': gpu_details(),\n        \'cpu_percent\': psutil.cpu_percent(interval=None),\n        \'memory\': {\'total_gb\': round(ram.total / 2**30, 2), \'used_gb\': round(ram.used / 2**30, 2), \'available_gb\': round(ram.available / 2**30, 2)},\n        \'disk\': {\'free_gb\': round(disk.free / 2**30, 2), \'total_gb\': round(disk.total / 2**30, 2)},\n        \'bridge_dir\': str(BRIDGE_DIR),\n        \'persistent_state_dir\': str(PERSIST_DIR) if PERSIST_DIR else None,\n        \'active_job_ids\': running,\n        \'recent_jobs\': [{k: j.get(k) for k in (\'job_id\', \'kind\', \'status\', \'exit_code\', \'started_at\', \'finished_at\', \'command\',\'progress\')}\n                        for j in map(job_with_progress,recent)],\n        \'events\': events,\n    }\n\n\ndef persist_snapshot():\n    snapshot = status()\n    put_json(STATE_DIR / \'last_status.json\', snapshot)\n    if PERSIST_DIR:\n        try:\n            PERSIST_DIR.mkdir(parents=True, exist_ok=True)\n            put_json(PERSIST_DIR / \'last_status.json\', snapshot)\n            with LOCK:\n                jobs = list(JOBS.values())[-30:]\n            put_json(PERSIST_DIR / \'jobs.json\', jobs)\n            for job in jobs:\n                source = JOB_DIR / job[\'job_id\'] / \'output.log\'\n                if source.exists():\n                    target = PERSIST_DIR / \'logs\' / f"{job[\'job_id\']}.log"\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with source.open(\'rb\') as reader:\n                        reader.seek(max(0, source.stat().st_size - MAX_LOG_BYTES))\n                        data = reader.read()\n                    target.write_bytes(data)\n            (PERSIST_DIR / \'heartbeat.txt\').write_text(now(), encoding=\'utf-8\')\n        except Exception as exc:\n            event(\'persistence_error\', detail=f\'{type(exc).__name__}: {exc}\'[:300])\n    return snapshot\n\n\ndef background_snapshots():\n    while True:\n        try:\n            persist_snapshot()\n        except Exception as exc:\n            event(\'snapshot_error\', detail=f\'{type(exc).__name__}: {exc}\'[:300])\n        time.sleep(15)\n\n\ndef wait_for_exit(job_id: str, process: subprocess.Popen):\n    rc = process.wait()\n    with LOCK:\n        j = JOBS[job_id]\n        j[\'status\'] = \'cancelled\' if j.get(\'cancel_requested\') else (\'succeeded\' if rc == 0 else \'failed\')\n        j[\'exit_code\'] = rc\n        j[\'finished_at\'] = now()\n        put_json(JOB_DIR / job_id / \'state.json\', j)\n        PROCESSES.pop(job_id, None)\n        event(\'job_finished\', job_id=job_id, exit_code=rc, state=j[\'status\'])\n    try:\n        persist_snapshot()\n    except Exception:\n        pass\n\n\ndef launch(command: str, kind: str, cwd: str, timeout: int, python_code: Optional[str] = None):\n    job_id = str(uuid.uuid4())\n    workdir = pathlib.Path(cwd).expanduser().resolve()\n    if not workdir.is_dir():\n        raise HTTPException(status_code=400, detail=f\'Working directory not found: {workdir}\')\n    folder = JOB_DIR / job_id\n    folder.mkdir(parents=True, exist_ok=True)\n    if kind == \'python\':\n        script = folder / \'task.py\'\n        script.write_text(python_code or \'\', encoding=\'utf-8\')\n        argv = [sys.executable, \'-u\', str(script)]\n        command_summary = f\'python {script} (code length {len(python_code or "")})\'\n    else:\n        argv = [\'/bin/bash\', \'-lc\', command]\n        command_summary = command[:700]\n    job = {\n        \'job_id\': job_id,\n        \'kind\': kind,\n        \'command\': command_summary,\n        \'cwd\': str(workdir),\n        \'started_at\': now(),\n        \'finished_at\': None,\n        \'status\': \'running\',\n        \'exit_code\': None,\n        \'timeout_seconds\': timeout,\n    }\n    try:\n        output_handle = (folder / \'output.log\').open(\'wb\')\n        process = subprocess.Popen(argv, cwd=str(workdir), stdout=output_handle,\n                                   stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,\n                                   start_new_session=True, bufsize=0,\n                                   env=job_environment())\n        output_handle.close()\n    except Exception as exc:\n        raise HTTPException(status_code=500, detail=f\'Failed to start: {exc}\')\n    with LOCK:\n        JOBS[job_id] = job\n        PROCESSES[job_id] = process\n        put_json(folder / \'state.json\', job)\n        event(\'job_started\', job_id=job_id, kind=kind)\n    threading.Thread(target=wait_for_exit, args=(job_id, process), daemon=True).start()\n    if timeout > 0:\n        def limit():\n            try:\n                process.wait(timeout=timeout)\n            except subprocess.TimeoutExpired:\n                with LOCK:\n                    if job_id in JOBS:\n                        JOBS[job_id][\'timeout_reached\'] = True\n                try:\n                    os.killpg(process.pid, signal.SIGTERM)\n                except ProcessLookupError:\n                    pass\n        threading.Thread(target=limit, daemon=True).start()\n    return job\n\n\nclass CommandRequest(BaseModel):\n    command: str = Field(min_length=1, max_length=100000)\n    cwd: str = \'/content\'\n    timeout_seconds: int = Field(default=3600, ge=0, le=86400)\n\n\nclass PythonRequest(BaseModel):\n    code: str = Field(min_length=1, max_length=2000000)\n    cwd: str = \'/content\'\n    timeout_seconds: int = Field(default=3600, ge=0, le=86400)\n\n\nclass WriteRequest(BaseModel):\n    path: str\n    content_base64: str\n    create_parents: bool = True\n\nclass ChunkRequest(BaseModel):\n    path: str\n    content_base64: str\n    offset: int = Field(ge=0)\n\nclass CommitRequest(BaseModel):\n    temporary: str\n    path: str\n    sha256: str = Field(pattern=\'^[0-9a-f]{64}$\')\n    size: int = Field(ge=0)\n\nclass TerminalWrite(BaseModel):\n    text: str = Field(max_length=100000)\n\n@app.get(\'/engine/capabilities\',dependencies=[Depends(secure)])\ndef engine_capabilities():\n    path=ENGINE_ROOT/\'runtime-capabilities.json\'\n    return json.loads(path.read_text()) if path.is_file() else {\'ready\':False,\'root\':str(ENGINE_ROOT)}\n\n@app.get(\'/files/stat\',dependencies=[Depends(secure)])\ndef file_stat(path: str):\n    p=pathlib.Path(path).expanduser().resolve()\n    if not p.is_file():raise HTTPException(status_code=404,detail=\'File not found\')\n    with p.open(\'rb\') as stream:digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\n    return {\'path\':str(p),\'size_bytes\':p.stat().st_size,\'sha256\':digest}\n\n@app.get(\'/files/read_chunk\',dependencies=[Depends(secure)])\ndef read_chunk(path: str,offset:int=Query(default=0,ge=0),length:int=Query(default=3000000,ge=1,le=3000000)):\n    p=pathlib.Path(path).expanduser().resolve()\n    if not p.is_file():raise HTTPException(status_code=404,detail=\'File not found\')\n    with p.open(\'rb\') as stream:stream.seek(offset);data=stream.read(length)\n    return {\'content_base64\':base64.b64encode(data).decode(),\'next_offset\':offset+len(data),\'size_bytes\':p.stat().st_size}\n\n@app.post(\'/files/write_chunk\',dependencies=[Depends(secure)])\ndef write_chunk(req:ChunkRequest):\n    try:data=base64.b64decode(req.content_base64,validate=True)\n    except Exception:raise HTTPException(status_code=400,detail=\'Invalid base64\')\n    if len(data)>3000000:raise HTTPException(status_code=413,detail=\'Chunk exceeds 3 MB\')\n    p=pathlib.Path(req.path).expanduser().resolve();p.parent.mkdir(parents=True,exist_ok=True)\n    with LOCK:\n        size=p.stat().st_size if p.exists() else 0\n        if size!=req.offset:raise HTTPException(status_code=409,detail=\'Upload offset differs from current file size\')\n        with p.open(\'ab\') as stream:stream.write(data)\n    return {\'next_offset\':req.offset+len(data)}\n\n@app.post(\'/files/commit\',dependencies=[Depends(secure)])\ndef commit_file(req:CommitRequest):\n    p=pathlib.Path(req.temporary).expanduser().resolve();target=pathlib.Path(req.path).expanduser().resolve()\n    if not p.is_file():raise HTTPException(status_code=404,detail=\'Temporary upload not found\')\n    with p.open(\'rb\') as stream:digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\n    if p.stat().st_size!=req.size or digest!=req.sha256:raise HTTPException(status_code=400,detail=\'Upload hash/size mismatch\')\n    target.parent.mkdir(parents=True,exist_ok=True);p.replace(target)\n    return {\'path\':str(target),\'sha256\':digest,\'size_bytes\':req.size}\n\n@app.post(\'/terminals\',dependencies=[Depends(secure)])\ndef start_terminal():\n    if os.name!=\'posix\':raise HTTPException(status_code=501,detail=\'PTY terminals require Linux\')\n    import pty\n    master,slave=pty.openpty();env=job_environment();env.update(TERM=\'xterm-256color\',PS1=\'gpu-runtime> \')\n    process=subprocess.Popen([\'/bin/bash\',\'--noprofile\',\'--norc\',\'-i\'],stdin=slave,stdout=slave,stderr=slave,start_new_session=True,env=env,cwd=str(ENGINE_ROOT if ENGINE_ROOT.is_dir() else BRIDGE_DIR))\n    os.close(slave);identifier=str(uuid.uuid4());folder=BRIDGE_DIR/\'terminals\'/identifier;folder.mkdir(parents=True)\n    entry={\'process\':process,\'master\':master,\'log\':folder/\'output.log\'}\n    with LOCK:TERMINALS[identifier]=entry\n    def read_output():\n        with entry[\'log\'].open(\'ab\',buffering=0) as output:\n            try:\n                while data:=os.read(master,4096):output.write(data)\n            except OSError:pass\n        process.wait()\n        try:os.close(master)\n        except OSError:pass\n    threading.Thread(target=read_output,daemon=True).start()\n    return {\'terminal_id\':identifier,\'pid\':process.pid}\n\n@app.get(\'/terminals/{identifier}/read\',dependencies=[Depends(secure)])\ndef read_terminal(identifier:str,offset:int=Query(default=0,ge=0),limit:int=Query(default=30000,ge=1,le=MAX_LOG_BYTES)):\n    with LOCK:entry=TERMINALS.get(identifier)\n    if entry is None:raise HTTPException(status_code=404,detail=\'Unknown terminal\')\n    data=b\'\'\n    if entry[\'log\'].is_file():\n        with entry[\'log\'].open(\'rb\') as stream:stream.seek(offset);data=stream.read(limit)\n    return {\'text\':data.decode(\'utf-8\',errors=\'replace\'),\'next_offset\':offset+len(data),\'running\':entry[\'process\'].poll() is None}\n\n@app.post(\'/terminals/{identifier}/write\',dependencies=[Depends(secure)])\ndef write_terminal(identifier:str,req:TerminalWrite):\n    with LOCK:entry=TERMINALS.get(identifier)\n    if entry is None:raise HTTPException(status_code=404,detail=\'Unknown terminal\')\n    if entry[\'process\'].poll() is not None:raise HTTPException(status_code=409,detail=\'Terminal has exited\')\n    try:os.write(entry[\'master\'],req.text.encode())\n    except OSError:raise HTTPException(status_code=409,detail=\'Terminal is closed\')\n    return {\'ok\':True}\n\n@app.post(\'/terminals/{identifier}/close\',dependencies=[Depends(secure)])\ndef close_terminal(identifier:str):\n    with LOCK:entry=TERMINALS.get(identifier)\n    if entry is None:raise HTTPException(status_code=404,detail=\'Unknown terminal\')\n    if entry[\'process\'].poll() is None:\n        try:os.killpg(entry[\'process\'].pid,signal.SIGHUP)\n        except ProcessLookupError:pass\n    return {\'ok\':True}\n\n\n@app.get(\'/health\', dependencies=[Depends(secure)])\ndef health():\n    return {\'status\': \'ok\', \'timestamp_utc\': now(), \'api_version\': 3}\n\n@app.on_event(\'shutdown\')\ndef shutdown_children():\n    with LOCK:\n        children=list(PROCESSES.values())+[entry[\'process\'] for entry in TERMINALS.values()]\n        for job in JOBS.values():\n            if job[\'status\']==\'running\':job[\'cancel_requested\']=True\n    for process in children:\n        if process.poll() is None:\n            try:\n                if os.name==\'posix\':os.killpg(process.pid,signal.SIGTERM)\n                else:process.terminate()\n            except ProcessLookupError:pass\n\n\n@app.get(\'/status\', dependencies=[Depends(secure)])\ndef read_status():\n    s = status()\n    put_json(STATE_DIR / \'last_status.json\', s)\n    return s\n\n\n@app.post(\'/jobs/command\', dependencies=[Depends(secure)])\ndef command_job(req: CommandRequest):\n    return launch(req.command, \'shell\', req.cwd, req.timeout_seconds)\n\n\n@app.post(\'/jobs/python\', dependencies=[Depends(secure)])\ndef python_job(req: PythonRequest):\n    return launch(\'\', \'python\', req.cwd, req.timeout_seconds, python_code=req.code)\n\n\n@app.get(\'/jobs\', dependencies=[Depends(secure)])\ndef list_jobs(limit: int = Query(default=20, ge=1, le=100)):\n    with LOCK:\n        return [job_with_progress(j) for j in list(JOBS.values())[-limit:]]\n\n\n@app.get(\'/jobs/{job_id}\', dependencies=[Depends(secure)])\ndef get_job(job_id: str):\n    with LOCK:\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail=\'Unknown job ID\')\n        return job_with_progress(JOBS[job_id])\n\n\n@app.get(\'/jobs/{job_id}/output\', dependencies=[Depends(secure)])\ndef job_output(job_id: str, offset: int = Query(default=0, ge=0), limit: int = Query(default=30000, ge=1, le=MAX_LOG_BYTES)):\n    with LOCK:\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail=\'Unknown job ID\')\n        job = job_with_progress(JOBS[job_id])\n    p = JOB_DIR / job_id / \'output.log\'\n    size = p.stat().st_size if p.exists() else 0\n    with p.open(\'rb\') as f:\n        f.seek(min(offset, size))\n        chunk = f.read(limit)\n    return {\'job\': job, \'text\': chunk.decode(\'utf-8\', errors=\'replace\'),\n            \'offset\': offset, \'next_offset\': min(offset, size) + len(chunk),\n            \'total_bytes\': size, \'has_more\': min(offset, size) + len(chunk) < size}\n\n\n@app.post(\'/jobs/{job_id}/cancel\', dependencies=[Depends(secure)])\ndef cancel_job(job_id: str):\n    with LOCK:\n        process = PROCESSES.get(job_id)\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail=\'Unknown job ID\')\n        if not process or process.poll() is not None:\n            return {\'ok\': False, \'detail\': \'Job already finished\'}\n        JOBS[job_id][\'cancel_requested\'] = True\n    try:\n        os.killpg(process.pid, signal.SIGTERM)\n    except ProcessLookupError:\n        pass\n    return {\'ok\': True, \'job_id\': job_id}\n\n\n@app.get(\'/files/list\', dependencies=[Depends(secure)])\ndef files_list(path: str = \'/content\', limit: int = Query(default=100, ge=1, le=500)):\n    p = pathlib.Path(path).expanduser().resolve()\n    if not p.is_dir():\n        raise HTTPException(status_code=400, detail=\'Not a directory\')\n    try:\n        items = sorted(p.iterdir(), key=lambda f: (not f.is_dir(), f.name.lower()))[:limit]\n        return {\'path\': str(p), \'items\': [{\'name\': x.name, \'is_dir\': x.is_dir(), \'size_bytes\': x.stat().st_size if x.is_file() else None}\n                                           for x in items]}\n    except OSError as exc:\n        raise HTTPException(status_code=400, detail=str(exc))\n\n\n@app.get(\'/files/read\', dependencies=[Depends(secure)])\ndef files_read(path: str):\n    p = pathlib.Path(path).expanduser().resolve()\n    if not p.is_file():\n        raise HTTPException(status_code=404, detail=\'File not found\')\n    size = p.stat().st_size\n    if size > MAX_FILE:\n        raise HTTPException(status_code=413, detail=\'File too large for direct transfer; use shell commands or chunked transfer\')\n    return {\'path\': str(p), \'content_base64\': base64.b64encode(p.read_bytes()).decode(\'ascii\'), \'size_bytes\': size}\n\n\n@app.post(\'/files/write\', dependencies=[Depends(secure)])\ndef files_write(req: WriteRequest):\n    try:\n        data = base64.b64decode(req.content_base64, validate=True)\n    except Exception:\n        raise HTTPException(status_code=400, detail=\'Invalid base64 data\')\n    if len(data) > MAX_FILE:\n        raise HTTPException(status_code=413, detail=\'File too large for direct transfer\')\n    p = pathlib.Path(req.path).expanduser().resolve()\n    if req.create_parents:\n        p.parent.mkdir(parents=True, exist_ok=True)\n    p.write_bytes(data)\n    return {\'path\': str(p), \'size_bytes\': len(data)}\n\n\n@app.get(\'/recovery\', dependencies=[Depends(secure)])\ndef recovery():\n    # This endpoint is useful after transient tunnel failure while VM lives.\n    with LOCK:\n        recent = list(JOBS.values())[-20:]\n        events = list(EVENTS)\n    log_tails = {}\n    for j in recent[-8:]:\n        p = JOB_DIR / j[\'job_id\'] / \'output.log\'\n        if p.exists():\n            with p.open(\'rb\') as f:\n                f.seek(max(0, p.stat().st_size - 2000))\n                log_tails[j[\'job_id\']] = f.read().decode(\'utf-8\', errors=\'replace\')\n    return {\'status\': status(), \'recent_jobs\': recent, \'events\': events, \'recent_output_tails\': log_tails}\n\n\ndef restore_local_history():\n    for folder in JOB_DIR.glob(\'*/state.json\'):\n        try:\n            old = json.loads(folder.read_text(\'utf-8\'))\n            if old.get(\'status\') == \'running\':\n                old[\'status\'] = \'interrupted_or_unknown\'\n                old[\'finished_at\'] = now()\n            JOBS[old[\'job_id\']] = old\n        except Exception:\n            pass\n    event(\'bridge_started\', restored_jobs=len(JOBS))\n\n\nrestore_local_history()\nthreading.Thread(target=background_snapshots, daemon=True).start()\n'

data=base64.b64decode(PAYLOAD)
if hashlib.sha256(data).hexdigest()!=EXPECTED_SHA256:raise ValueError('Runtime tools package hash mismatch')
destination=pathlib.Path('/content').resolve()
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    if any(not (destination/name).resolve().is_relative_to(destination) for name in archive.namelist()):raise ValueError('Unsafe runtime archive')
    archive.extractall(destination)
del PAYLOAD,data
engine=destination/'video_engine';base=destination/'colab_codex_bridge';base.mkdir(exist_ok=True)
(base/'bridge_server.py').write_text(server_source,encoding='utf-8')
private=base/'bridge_private.json';saved=json.loads(private.read_text()) if private.is_file() else {}
token=saved.get('token');port=saved.get('port');server_pid=saved.get('server_pid')
reuse=False
if token and port and server_pid and psutil.pid_exists(server_pid):
    try:
        answer=requests.get(f'http://127.0.0.1:{port}/health',headers={'Authorization':'Bearer '+token},timeout=3)
        reuse=answer.status_code==200 and answer.json().get('api_version')==3
    except requests.RequestException:pass
if not reuse:
    if server_pid and psutil.pid_exists(server_pid):
        process=psutil.Process(server_pid)
        if 'bridge_server:app' in ' '.join(process.cmdline()):process.terminate();process.wait(timeout=10)
    token=secrets.token_urlsafe(48)
    with socket.socket() as sock:sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
    env=os.environ.copy();env.update(COLAB_BRIDGE_TOKEN=token,COLAB_BRIDGE_DIR=str(base),VIDEO_ENGINE_ROOT=str(engine))
    with (base/'server.log').open('ab',buffering=0) as output:
        server=subprocess.Popen([sys.executable,'-m','uvicorn','bridge_server:app','--host','127.0.0.1','--port',str(port),'--no-access-log'],cwd=base,env=env,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    server_pid=server.pid
    for attempt in range(30):
        if server.poll() is not None:raise RuntimeError('Runtime server exited. Inspect /content/colab_codex_bridge/server.log')
        try:
            if requests.get(f'http://127.0.0.1:{port}/health',headers={'Authorization':'Bearer '+token},timeout=2).status_code==200:break
        except requests.RequestException:pass
        time.sleep(1)
    else:raise RuntimeError('Runtime server did not become ready')
old_tunnel=saved.get('tunnel_pid')
if old_tunnel and psutil.pid_exists(old_tunnel):
    process=psutil.Process(old_tunnel)
    if 'cloudflared' in ' '.join(process.cmdline()):process.terminate()
cloudflared=base/'cloudflared'
if not cloudflared.is_file():
    if platform.machine().lower() not in ('x86_64','amd64'):raise RuntimeError('This launcher expects Linux x86_64')
    request=urllib.request.Request('https://api.github.com/repos/cloudflare/cloudflared/releases/latest',headers={'User-Agent':'VideoEngine/1.0'})
    metadata=json.loads(urllib.request.urlopen(request,timeout=30).read())
    asset=next(a for a in metadata['assets'] if a['name']=='cloudflared-linux-amd64')
    urllib.request.urlretrieve(asset['browser_download_url'],cloudflared)
    if asset.get('digest','').startswith('sha256:') and 'sha256:'+hashlib.sha256(cloudflared.read_bytes()).hexdigest()!=asset['digest']:raise ValueError('Cloudflared publisher digest mismatch')
    if cloudflared.read_bytes()[:4]!=b'\x7fELF':raise ValueError('Cloudflared is not a Linux executable')
    cloudflared.chmod(0o700)
tunnel=subprocess.Popen([str(cloudflared),'tunnel','--no-autoupdate','--protocol','http2','--url',f'http://127.0.0.1:{port}'],cwd=base,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
address={'url':None}
def consume():
    with (base/'tunnel.log').open('a') as output:
        for line in tunnel.stdout:
            output.write(line);output.flush()
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
            if match:address['url']=match.group(0)
threading.Thread(target=consume,daemon=True).start()
for attempt in range(60):
    if address['url']:break
    if tunnel.poll() is not None:raise RuntimeError('Tunnel exited; inspect /content/colab_codex_bridge/tunnel.log')
    time.sleep(1)
else:raise RuntimeError('The tunnel did not return a URL')
private.write_text(json.dumps({'token':token,'port':port,'server_pid':server_pid,'tunnel_pid':tunnel.pid}));private.chmod(0o600)
print('\nGPU RUNTIME BRIDGE ONLINE',flush=True)
print('BRIDGE URL:',address['url'],flush=True)
print('PRIVATE TOKEN:',token,flush=True)
print('Keep the token private. This is a general runtime, not a prepared video.',flush=True)
print('Tools: Python, shell jobs, interactive terminal, large file upload/download, GPU project builds/probes/renders.',flush=True)
print('Preparing cached video tools; no scene or video is rendered during setup.',flush=True)
headers={'Authorization':'Bearer '+token}
setup_code='import subprocess,sys;subprocess.run([sys.executable,'+repr(str(engine/'runtime_setup.py'))+',"--root",'+repr(str(engine))+'],check=True)'
job=requests.post(f'http://127.0.0.1:{port}/jobs/python',headers=headers,json={'code':setup_code,'cwd':'/content','timeout_seconds':1800},timeout=10).json()
print('SETUP JOB:',job['job_id'],flush=True)
offset=0
while True:
    answer=requests.get(f'http://127.0.0.1:{port}/jobs/{job["job_id"]}/output',headers=headers,params={'offset':offset,'limit':160000},timeout=10).json()
    for line in answer['text'].splitlines():
        if line.startswith(('RUNTIME_','BUILD_STAGE','Traceback','RuntimeError')):print(line,flush=True)
    offset=answer['next_offset']
    if answer['job']['status']!='running':
        print('VIDEO TOOLS READY' if answer['job'].get('exit_code')==0 else 'Video setup failed; bridge remains available for diagnostics.',flush=True)
        break
    time.sleep(3)
print('Bridge is listening. Keep this cell running; stop it to close the endpoint.',flush=True)
sys.path.insert(0,str(engine))
from runtime_dashboard import render_dashboard
from IPython.display import HTML,display
dashboard=display(HTML(render_dashboard([])),display_id=True)
try:
    while psutil.pid_exists(server_pid) and tunnel.poll() is None:
        try:
            response=requests.get(f'http://127.0.0.1:{port}/jobs',headers=headers,timeout=10)
            response.raise_for_status()
            dashboard.update(HTML(render_dashboard(response.json())))
        except requests.RequestException:
            dashboard.update(HTML('<p>Progress connection interrupted; retrying. Existing job logs are retained.</p>'))
        time.sleep(3)
    print('Bridge server or tunnel stopped. Rerun this cell to reconnect.',flush=True)
except KeyboardInterrupt:
    tunnel.terminate()
    if psutil.pid_exists(server_pid):psutil.Process(server_pid).terminate()
    print('Runtime bridge stopped.',flush=True)
